In [1]:
from collections import defaultdict
import itertools
from nltk import ngrams
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize
import numpy as np
import pandas as pd
import pickle
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Union, Optional

from dap_job_quality import config, PROJECT_DIR, logging
from dap_job_quality.getters.afs_data import get_eyp_ads, get_sim_occ_ads

# Load BERT model and tokenizer
model_name = config["sentence_model"]

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

/Users/rosie.oxbury/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2024-06-13 17:29:50,267 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


In [2]:
# Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[
        0
    ]  # First element of model_output contains all token embeddings
    input_mask_expanded = (
        attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    )
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask


def embed_sentences(
    sentences: List[str], model_name: str = model_name
) -> List[torch.Tensor]:
    """
    Generate embeddings for each sentence in a list of sentences using a specified model.

    Follows the method described here: https://www.sbert.net/examples/applications/computing-embeddings/README.html

    Args:
        sentences (List[str]): A list of sentences to be embedded.
        model_name (str): The name of the model to use for generating embeddings. Default
                          is a globally defined variable `SENT_MODEL`.

    Returns:
        List[torch.Tensor]: A list of tensors where each tensor represents the embedding
                            of a corresponding sentence in the input list.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    # Tokenize sentences
    encoded_input = tokenizer(
        sentences, padding=True, truncation=True, max_length=512, return_tensors="pt"
    )

    # Compute token embeddings
    with torch.no_grad():
        model_output = model(**encoded_input)

    # Perform pooling. In this case, mean pooling
    embeddings = mean_pooling(model_output, encoded_input["attention_mask"])

    return embeddings

def split_ngrams(text, length=6, n=4):
    if len(text.split()) > length:
        ngram_list = list(ngrams(text.split(), n))
    else:
        ngram_list = [text]
    return ngram_list

In [3]:
eyp = get_eyp_ads()#[['id', 'clean_description']]
sim_occs = get_sim_occ_ads()#[['id', 'clean_description']]
all_job_ads = pd.concat([eyp, sim_occs], axis=0).drop_duplicates()

lookup = pd.read_csv(PROJECT_DIR / "inputs/keyword_lookup - v5.csv")

In [4]:
logging.info(len(all_job_ads))
all_job_ads = all_job_ads[all_job_ads['created']>='2022-01-01']
job_ads_sample = all_job_ads.sample(1000, random_state=42)
logging.info(len(all_job_ads))

2024-06-13 17:30:18,333 - root - INFO - 254529
2024-06-13 17:30:18,376 - root - INFO - 172263


In [5]:
all_job_ads['sentences'] = all_job_ads['clean_description'].apply(sent_tokenize)
all_job_ads = all_job_ads.explode('sentences')

In [6]:
with open(PROJECT_DIR / 'outputs/models/sentence_classifier/logistic_regression.pkl', 'rb') as file:
    model = pickle.load(file)

# Load the saved PCA transformer (if used)
with open(PROJECT_DIR / 'outputs/models/sentence_classifier/pca.pkl', 'rb') as file:
    pca = pickle.load(file)

In [7]:
ad_embeddings = embed_sentences(all_job_ads['sentences'].tolist())

Some weights of BertModel were not initialized from the model checkpoint at jjzha/jobbert-base-cased and are newly initialized: ['bert.pooler.dense.weight', 'bert.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


: 

In [ ]:
X_new_pca = pca.transform(ad_embeddings)

predictions = model.predict(X_new_pca)

In [ ]:
all_job_ads['ngrams'] = all_job_ads['sentences'].apply(lambda x: split_ngrams(x, 6, 4))
sentence_df_long = all_job_ads.explode('ngrams')
sentence_df_long['ngrams'] = sentence_df_long['ngrams'].apply(lambda x: ' '.join(x) if isinstance(x, tuple) else x)
sentence_df_long.head()

In [ ]:
unique_ngrams = list(sentence_df_long['ngrams'].unique())

In [ ]:
# Embed the target phrases
target_embeddings = embed_sentences(target_phrases)
    
ngram_embeddings = embed_sentences(unique_ngrams)
        
similarities = calculate_cosine_similarity(ngram_embeddings, target_embeddings)

similarities
        
matches = defaultdict(list)
for i, ngram in enumerate(unique_ngrams):
    for j, target_phrase in enumerate(target_phrases):
        if similarities[i, j] > 0.8:
            matches[ngram].append((target_phrase, similarities[i, j]))
            
# Deduplicate target phrases for each text
deduplicated_matches = {}
for ngram, matches_list in matches.items():
    unique_matches = list({phrase: sim for phrase, sim in matches_list}.items())
    deduplicated_matches[ngram] = unique_matches

In [ ]:
matches_df = pd.DataFrame(deduplicated_matches.items(), columns=['ngram', 'matches'])

In [ ]:
sentence_df_long = pd.merge(sentence_df_long, matches_df, how='left', left_on='ngrams', right_on='ngram')
sentence_df_long.head()